# H3d — Retaliation

**H3d**  politicians who have been accused of lying are more likely to
subsequently accuse their accuser of lying in return.

### Why this needs a different design

H3a–H3c are already answered by the speaker-year models in `02_populism`. H3d is
not, because it is about a *directed pair over time*, not an individual.

The trap is **dyadic propensity**. Two front-benchers who spar constantly will
accuse each other often in both directions, with no retaliation involved. A raw
correlation between "A accused B" and "B accused A" would pick that up and call
it retaliation.

The fix is to stay **inside the dyad and use time order**: for a given pair,
is B→A more likely in the months *following* an A→B than in that same pair's
other months? Dyad fixed effects absorb "these two fight a lot"; only the timing
identifies retaliation.

### Design

- **Unit**: directed dyad-month. For direction A→B, one row per month the pair is
  observable.
- **Outcome**: did A accuse B in this month?
- **Predictor**: did B accuse A in the previous `LAGS` months?
- **Estimator**: conditional logit with **dyad-direction fixed effects**, SEs
  clustered by unordered dyad.
- **Baseline**: permutation — reshuffle targets within country-year and re-run,
  to show the effect exceeds what dyadic sorting alone produces.

Two things the feasibility check flagged, both handled below: immediate
same-month exchanges are separated from deliberate later retaliation, and
interjections (structurally attributed heckles) get a robustness run.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# ConditionalLogit is exposed on statsmodels.api in recent versions; fall back to
# the module path if this build does not expose it.
try:
    ConditionalLogit = sm.ConditionalLogit
except AttributeError:
    from statsmodels.discrete.conditional_models import ConditionalLogit

from lib import data, viz
from lib.codebooks import EXCLUDED_COUNTRIES
viz.apply_style()

LAGS = 3            # months back over which a prior accusation counts as a trigger
MIN_MONTHS = 6      # dyad-directions observed for fewer months are dropped
N_PERM = 200        # permutation draws (section 4; the slow part)
SEED = 20260810

rng = np.random.default_rng(SEED)
print(f"statsmodels {sm.__version__}, ConditionalLogit ready")
print(f"excluded countries: {sorted(EXCLUDED_COUNTRIES)}")

## 1. Directed events

Person-targets where **both** accuser and target resolved to a known speaker,
dated, and not self-accusations.

In [ ]:
con = data.duck()

ev = con.execute("""
    SELECT id, country, source_dataset, date,
           accuser_speaker_id AS a,
           target_speaker_id  AS b,
           COALESCE(is_interjection, 0) AS is_interjection
    FROM accusations
    WHERE target_type = 'person'
      AND accuser_speaker_id IS NOT NULL
      AND target_speaker_id  IS NOT NULL
      AND accuser_speaker_id <> target_speaker_id
      AND date IS NOT NULL AND length(date) >= 7
""").df()

ev["date"] = pd.to_datetime(ev["date"].str[:10], errors="coerce")
ev = ev.dropna(subset=["date"])
ev["month"] = ev["date"].values.astype("datetime64[M]")

print(f"usable directed events : {len(ev):,}")
print(f"distinct accusers      : {ev['a'].nunique():,}")
print(f"distinct targets       : {ev['b'].nunique():,}")
print(f"interjections          : {int(ev['is_interjection'].sum()):,}")
print(f"date range             : {ev['date'].min():%Y-%m} to {ev['date'].max():%Y-%m}")

In [ ]:
# directed and reciprocal dyads
pairs = ev.groupby(["a", "b"]).size().rename("n").reset_index()
pairset = set(zip(pairs["a"], pairs["b"]))
recip = {(a, b) for (a, b) in pairset if (b, a) in pairset}

print(f"directed dyads (A->B)          : {len(pairs):,}")
print(f"reciprocal directed dyads      : {len(recip):,}")
print(f"  = unordered reciprocal pairs : {len(recip)//2:,}")

cnt = dict(zip(zip(pairs["a"], pairs["b"]), pairs["n"]))
both2 = sum(1 for (a, b) in recip
            if a < b and cnt.get((a, b), 0) >= 2 and cnt.get((b, a), 0) >= 2)
print(f"  with >= 2 events each way    : {both2:,}")

In [ ]:
# --- how much of the reciprocity depends on interjections? -------------------
# Interjections are recorded as accuser = parsed heckler, target = the HOST
# speaker. The target is a structural assumption, not observed, so a wrong guess
# invents a dyad -- and since heckle-and-reply happen in the same sitting, they
# can manufacture reciprocity by construction.

ev_ni_d = ev[ev["is_interjection"] == 0]
ps_ni_d = set(zip(ev_ni_d["a"], ev_ni_d["b"]))
recip_ni = {(a, b) for (a, b) in ps_ni_d if (b, a) in ps_ni_d}

n_all = len(recip) // 2
n_ni = len(recip_ni) // 2
print(f"reciprocal pairs, all events           : {n_all:,}")
print(f"reciprocal pairs, no interjections     : {n_ni:,}")
print(f"pairs existing ONLY via interjections  : {n_all - n_ni:,} "
      f"({(n_all - n_ni) / max(n_all, 1) * 100:.1f}%)")

# do the two directions of a pair arrive on the SAME DAY? (the artefact signature)
ev_r_chk = ev[[(a, b) in recip for a, b in zip(ev["a"], ev["b"])]].copy()
ev_r_chk["pair"] = [tuple(sorted((a, b))) for a, b in
                    zip(ev_r_chk["a"], ev_r_chk["b"])]

two_way_days = int((ev_r_chk.groupby(["pair", "date"])["a"].nunique() > 1).sum())
heckle_days = int(ev_r_chk[ev_r_chk["is_interjection"] == 1]
                  .groupby(["pair", "date"]).ngroups)

print(f"\npair-days with BOTH directions present : {two_way_days:,}")
print(f"pair-days involving a heckle           : {heckle_days:,}")
print("\nSame-day two-way exchanges are what `attacked_same_month` absorbs.")
print("The LAGGED term is the H3d result and should survive their removal;")
print("robustness (a) below re-runs it with interjections dropped entirely.")

## 2. Dyad-month panel

For every reciprocal pair, both directions get a row per month between the
pair's first and last observed interaction. Restricting to reciprocal pairs is
not a selection problem: conditional logit discards any group without variation
in the outcome anyway, so one-directional pairs contribute nothing regardless.

In [ ]:
ev_r = ev[[(a, b) in recip for a, b in zip(ev["a"], ev["b"])]].copy()
ev_r["pair"] = [tuple(sorted((a, b))) for a, b in zip(ev_r["a"], ev_r["b"])]
print(f"events inside reciprocal pairs: {len(ev_r):,}")

# observable span per unordered pair
span = ev_r.groupby("pair")["month"].agg(["min", "max"])

rows = []
for pair, (m0, m1) in span.iterrows():
    months = pd.date_range(m0, m1, freq="MS").values.astype("datetime64[M]")
    if len(months) < MIN_MONTHS:
        continue
    a, b = pair
    for src, dst in ((a, b), (b, a)):
        for m in months:
            rows.append((f"{src}>{dst}", f"{a}|{b}", src, dst, m))

panel_df = pd.DataFrame(rows, columns=["direction", "pair", "src", "dst", "month"])
print(f"dyad-month rows: {len(panel_df):,} "
      f"({panel_df['direction'].nunique():,} dyad-directions, "
      f"{panel_df['pair'].nunique():,} pairs)")

In [ ]:
# outcome: did src accuse dst this month?
made = (ev_r.assign(direction=lambda d: d["a"] + ">" + d["b"])
            .groupby(["direction", "month"]).size().rename("n_acc").reset_index())
panel_df = panel_df.merge(made, on=["direction", "month"], how="left")
panel_df["n_acc"] = panel_df["n_acc"].fillna(0)
panel_df["y"] = (panel_df["n_acc"] > 0).astype(int)

# same series for the REVERSE direction, to build the trigger
rev = panel_df[["direction", "pair", "src", "dst", "month", "y"]].copy()
rev["direction_rev"] = rev["dst"] + ">" + rev["src"]
rev = rev[["direction_rev", "month", "y"]].rename(
    columns={"direction_rev": "direction", "y": "y_rev"})
panel_df = panel_df.merge(rev, on=["direction", "month"], how="left")
panel_df["y_rev"] = panel_df["y_rev"].fillna(0)

panel_df = panel_df.sort_values(["direction", "month"]).reset_index(drop=True)

# trigger: was I accused by them in the previous LAGS months?
g = panel_df.groupby("direction", sort=False)["y_rev"]
lagsum = sum(g.shift(k).fillna(0) for k in range(1, LAGS + 1))
panel_df["attacked_recently"] = (lagsum > 0).astype(int)
panel_df["attacked_same_month"] = (panel_df["y_rev"] > 0).astype(int)

print(f"months where src accused dst          : {int(panel_df['y'].sum()):,}")
print(f"months preceded by being accused      : {int(panel_df['attacked_recently'].sum()):,}")
print(f"months with a same-month exchange     : {int(panel_df['attacked_same_month'].sum()):,}")

### Raw comparison

Before any model: how often does A accuse B in months that follow an attack,
versus months that do not?

In [ ]:
tab = panel_df.groupby("attacked_recently")["y"].agg(["mean", "sum", "size"])
tab.index = ["not recently attacked", "recently attacked"]
tab.columns = ["P(accuse)", "months with accusation", "months"]
print(tab.round(4).to_string())

ratio = tab.loc["recently attacked", "P(accuse)"] / tab.loc["not recently attacked", "P(accuse)"]
print(f"\nraw ratio: {ratio:.2f}x")
print("NOTE: this is NOT the test -- pairs that fight a lot inflate both cells.")
print("The dyad fixed effects below are what isolate retaliation.")

## 3. Main test — conditional logit with dyad-direction fixed effects

Each dyad-direction is its own stratum, so the estimate comes only from
*within-pair* variation over time. Groups with no variation in the outcome drop
out automatically, which is exactly the desired behaviour.

In [ ]:
def fit_clogit(d, xcols, group="direction"):
    """Conditional logit with fixed effects for `group`."""
    d = d.dropna(subset=xcols + ["y"])
    keep = d.groupby(group)["y"].transform(lambda s: 0 < s.sum() < len(s))
    d = d[keep]
    if d.empty:
        print("no within-group variation -- nothing to estimate")
        return None, d
    m = ConditionalLogit(d["y"], d[xcols], groups=d[group]).fit(disp=False)
    return m, d


def report_or(m, label):
    if m is None:
        return
    print(f"\n--- {label} ---")
    print(f"{'term':<26}{'OR':>8}{'95% CI':>22}{'p':>10}")
    ci = m.conf_int()
    for t in m.params.index:
        b = m.params[t]; lo, hi = ci.loc[t]
        print(f"{t:<26}{np.exp(b):>8.3f}"
              f"{f'[{np.exp(lo):.3f}, {np.exp(hi):.3f}]':>22}"
              f"{m.pvalues[t]:>10.3g}")


m_main, d_main = fit_clogit(panel_df, ["attacked_recently"])
print(f"strata contributing: {d_main['direction'].nunique():,} "
      f"of {panel_df['direction'].nunique():,}")
report_or(m_main, f"H3d: retaliation within {LAGS} months (dyad-direction FE)")
print("\nH3d expects OR > 1.")

### Separating immediate exchange from deliberate retaliation

A same-month exchange is usually one argument — a riposte inside a single debate,
and for interjections partly a coding artefact. Deliberate retaliation is the
lagged effect. Entering both distinguishes them.

In [ ]:
m_split, _ = fit_clogit(panel_df,
                        ["attacked_same_month", "attacked_recently"])
report_or(m_split, "same-month exchange vs. lagged retaliation")
print("\nThe lagged term surviving here is the real H3d result:")
print("it is retaliation across sessions, not a riposte inside one debate.")

## 4. Permutation baseline

Reassign each accusation's target at random among MPs accused in the same
country-year, rebuild the panel, and re-estimate. This preserves how much each
person accuses and is accused, but destroys who-targets-whom. The observed
coefficient should sit outside this distribution.

In [ ]:
def build_and_fit(events):
    """Rebuild the dyad-month panel from an event table and return the coefficient."""
    p = events.groupby(["a", "b"]).size().reset_index()
    ps = set(zip(p["a"], p["b"]))
    rc = {(a, b) for (a, b) in ps if (b, a) in ps}
    if not rc:
        return np.nan
    e = events[[(a, b) in rc for a, b in zip(events["a"], events["b"])]].copy()
    e["pair"] = [tuple(sorted((a, b))) for a, b in zip(e["a"], e["b"])]
    sp = e.groupby("pair")["month"].agg(["min", "max"])

    rws = []
    for pair, (m0, m1) in sp.iterrows():
        months = pd.date_range(m0, m1, freq="MS").values.astype("datetime64[M]")
        if len(months) < MIN_MONTHS:
            continue
        a, b = pair
        for s_, d_ in ((a, b), (b, a)):
            for m in months:
                rws.append((f"{s_}>{d_}", s_, d_, m))
    if not rws:
        return np.nan
    pn = pd.DataFrame(rws, columns=["direction", "src", "dst", "month"])

    mk = (e.assign(direction=lambda x: x["a"] + ">" + x["b"])
           .groupby(["direction", "month"]).size().rename("n_acc").reset_index())
    pn = pn.merge(mk, on=["direction", "month"], how="left")
    pn["y"] = (pn["n_acc"].fillna(0) > 0).astype(int)

    rv = pn[["src", "dst", "month", "y"]].copy()
    rv["direction"] = rv["dst"] + ">" + rv["src"]
    rv = rv[["direction", "month", "y"]].rename(columns={"y": "y_rev"})
    pn = pn.merge(rv, on=["direction", "month"], how="left")
    pn["y_rev"] = pn["y_rev"].fillna(0)
    pn = pn.sort_values(["direction", "month"])

    gg = pn.groupby("direction", sort=False)["y_rev"]
    ls = sum(gg.shift(k).fillna(0) for k in range(1, LAGS + 1))
    pn["attacked_recently"] = (ls > 0).astype(int)

    mm, _ = fit_clogit(pn, ["attacked_recently"])
    return np.nan if mm is None else mm.params["attacked_recently"]


obs = m_main.params["attacked_recently"]
perm = []
for i in range(N_PERM):
    sh = ev_r.copy()
    sh["b"] = (sh.groupby(["country", sh["date"].dt.year])["b"]
                 .transform(lambda s: rng.permutation(s.values)))
    sh = sh[sh["a"] != sh["b"]]
    perm.append(build_and_fit(sh))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_PERM} permutations", end="\r", flush=True)

perm = np.array([p for p in perm if np.isfinite(p)])
pval = float((perm >= obs).mean()) if len(perm) else np.nan
print(f"\nobserved coefficient : {obs:+.3f}  (OR {np.exp(obs):.2f})")
print(f"permutation mean     : {perm.mean():+.3f}  (n={len(perm)})")
print(f"permutation p-value  : {pval:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(np.exp(perm), bins=30, color="#9a9a9a", alpha=0.7,
        label="permuted (targets reshuffled)")
ax.axvline(np.exp(obs), color="#c0603f", lw=2.5, label="observed")
ax.axvline(1, color="black", lw=1, ls=":")
ax.set_xlabel("odds ratio for retaliation")
ax.set_ylabel("permutations")
ax.set_title("Retaliation effect against a who-targets-whom null")
ax.legend()
fig.tight_layout()
viz.savefig(fig, "h3d_permutation")
plt.show()

## 5. Robustness

In [ ]:
# (a) excluding interjections -- heckles are structurally attributed to the host
ev_ni = ev[ev["is_interjection"] == 0]
p_ni = ev_ni.groupby(["a", "b"]).size().reset_index()
ps_ni = set(zip(p_ni["a"], p_ni["b"]))
rc_ni = {(a, b) for (a, b) in ps_ni if (b, a) in ps_ni}
e_ni = ev_ni[[(a, b) in rc_ni for a, b in zip(ev_ni["a"], ev_ni["b"])]].copy()
b_ni = build_and_fit(e_ni)
print(f"excluding interjections : OR {np.exp(b_ni):.3f}  "
      f"({len(e_ni):,} events, {len(rc_ni)//2:,} pairs)")

# (b) dropping the three countries that dominate the sample
top3 = ev_r["country"].value_counts().head(3).index.tolist()
e_rest = ev_r[~ev_r["country"].isin(top3)]
b_rest = build_and_fit(e_rest)
print(f"excluding {top3} : OR {np.exp(b_rest):.3f}  ({len(e_rest):,} events)")

# (c) sensitivity to the lag window
for lag in (1, 3, 6, 12):
    gg = panel_df.groupby("direction", sort=False)["y_rev"]
    ls = sum(gg.shift(k).fillna(0) for k in range(1, lag + 1))
    tmp = panel_df.assign(attacked_recently=(ls > 0).astype(int))
    mm, _ = fit_clogit(tmp, ["attacked_recently"])
    if mm is not None:
        print(f"lag window {lag:>2} months  : OR {np.exp(mm.params['attacked_recently']):.3f}"
              f"  p={mm.pvalues['attacked_recently']:.3g}")

## 6. Verdict

| check | OR | p |
|---|---|---|
| Main (dyad-direction FE) | | |
| Lagged term, same-month controlled | | |
| Permutation p-value | | |
| Excluding interjections | | |
| Excluding top-3 countries | | |

**H3d is supported** if the lagged odds ratio is above 1, survives controlling
for same-month exchange, and sits outside the permutation distribution.

The strongest version of the claim needs all three: the raw ratio shows only that
antagonistic pairs exist; the fixed effects show the timing matters within a pair;
the permutation shows it exceeds what dyadic sorting alone produces.

Report the **lagged** coefficient, not the same-month one — a riposte inside a
single debate is a different phenomenon from the reputation-management mechanism
H3d actually describes, and for interjections it is partly a coding artefact.